# Part 4 - Running Your First AWS Bedrock Agent

*Last updated: November 2026*

Call the **Bedrock Agent** you built in the console (the course-recommendation agent whose action groups
use the Lambda functions in this folder). The notebook assumes an IAM role that is allowed to invoke the
agent, then sends the agent a message and prints the streamed response events.

## How to run this notebook

1. Set up the course environment once, in the course folder: `uv sync`.
   New to coding? Follow `BEGINNER_SETUP.md` step by step. Otherwise see `LOCAL_SETUP.MD`.
2. Select the **`.venv`** Python kernel (*Python 3.12*) for this notebook.
3. Copy `.env.example` (course folder) to `.env` and add your keys. The notebook finds `.env` automatically.

**Keys and settings used in this notebook**
- `AWS_REGION` (optional)
- `BEDROCK_AGENT_ID` (optional)
- `BEDROCK_AGENT_ALIAS_ID` (optional)
- `BEDROCK_INVOKE_ROLE_ARN` (optional)

Run the cells from top to bottom.

## Before you start (AWS prerequisites)

1. An AWS account. Bedrock calls are **billed to your account**.
2. The AWS CLI installed and signed in with **your own** IAM user: run `aws configure`.
   Give that user only the permissions it needs (for example the `AmazonBedrockFullAccess` managed policy)
   rather than administrator access.
3. In the Bedrock console, enable access to the model you want to use, in your region.
4. Put your settings in the course `.env` file (see `.env.example`). Nothing account-specific is written in this notebook.

### Values you need from **your** AWS account

| `.env` variable | Where to find it |
|---|---|
| `BEDROCK_AGENT_ID` | Bedrock console → Agents → your agent → *Agent ID* |
| `BEDROCK_AGENT_ALIAS_ID` | Your agent → *Aliases* → *Alias ID* |
| `BEDROCK_INVOKE_ROLE_ARN` | IAM → Roles → the role your user assumes to call the agent. Its trust policy is `AssumeRolepolicy.json.txt`; its permissions policy is `CallingAgentInlinePolicy.json.txt` (replace the placeholders in both files). |

> **Compatibility update:** the recording showed the instructor's own role ARN and agent IDs. They are now read from `.env`, so the notebook works with your account.

In [ ]:
import json
import os
import uuid

import boto3
from dotenv import load_dotenv

load_dotenv()
session_id = str(uuid.uuid4())

In [ ]:
REGION = os.getenv("AWS_REGION") or "us-east-1"
ROLE_ARN = os.getenv("BEDROCK_INVOKE_ROLE_ARN")      # arn:aws:iam::<your-account-id>:role/<your-role-name>
AGENT_ID = os.getenv("BEDROCK_AGENT_ID")
AGENT_ALIAS_ID = os.getenv("BEDROCK_AGENT_ALIAS_ID")

missing = [name for name, value in {
    "BEDROCK_INVOKE_ROLE_ARN": ROLE_ARN,
    "BEDROCK_AGENT_ID": AGENT_ID,
    "BEDROCK_AGENT_ALIAS_ID": AGENT_ALIAS_ID,
}.items() if not value]
print("Missing settings in .env:", ", ".join(missing)) if missing else print("Agent settings loaded.")

In [ ]:
sts_client = boto3.client("sts", region_name=REGION)

In [ ]:
assumed_role_object = sts_client.assume_role(
    RoleArn=ROLE_ARN, RoleSessionName="MySession"
)
credentials = assumed_role_object["Credentials"]

In [ ]:
# Create a new client using the assumed role credentials
bedrock_agent_runtime = boto3.client(
    "bedrock-agent-runtime",
    aws_access_key_id=credentials["AccessKeyId"],
    aws_secret_access_key=credentials["SecretAccessKey"],
    aws_session_token=credentials["SessionToken"],
    region_name=REGION
)

In [ ]:
# Input for the agent
input_text = """Hi I need help with choosing a course, can you help"""

In [ ]:
try:
    invoke_agent_response = bedrock_agent_runtime.invoke_agent(
        agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID, inputText=input_text, sessionId=session_id
    )
    event_stream = invoke_agent_response["completion"]

    for event in event_stream:
        print(event)

except Exception as e:
    print(f"Error: {e}")